# MAA — Eksperimen S1: baseline skeleton di CARR (tanpa augmentasi usulan)

Melatih **ST-GCN, ST-GCN++, CTR-GCN, PoseC3D** pada 9 protokol CARR (CS1–4, CL1–4, CG) × 5 seed = **180 unit**.
Augmentasi hanya augmentasi dasar benchmark CARR (flip, rotasi ±10°, skala 0,9–1,1) = varian **A0**.
Metrik: macro-F1 tingkat video, keseluruhan dan subgrup alat bantu (`dis`).

**Cara pakai:** *Runtime → GPU (T4 cukup)* → **Run all**. Buka notebook yang sama di sebanyak mungkin sesi Colab
(dua akun pun boleh) dan **Run all** di semuanya, tanpa mengubah apa pun. Tiap sesi mengambil unit secara acak dan
mengunci unit yang sedang dikerjakannya; unit selesai dilewati, unit terputus dilanjutkan. Kalau sesi putus, cukup Run all lagi.

**Syarat:** `MyDrive/CARR` (dataset + `cache/pose` dari benchmark skeleton). Untuk akun kedua: bagikan folder `CARR`
sebagai Editor, lalu di akun kedua *Shared with me → klik kanan CARR → Organize → Add shortcut → My Drive*.
Hasil ditulis ke `MyDrive/CARR/maa/`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Pengaturan

In [ ]:
DATA   = '/content/drive/MyDrive/CARR'      # dataset, cache/pose dan hasil (maa/) — sama dengan eksperimen sebelumnya

EXP    = 'S1'
AUGS   = ['A0']
MODELS = ['ST-GCN', 'ST-GCN++', 'CTR-GCN', 'PoseC3D']
SPLITS = ['CS1', 'CS2', 'CS3', 'CS4', 'CL1', 'CL2', 'CL3', 'CL4', 'CG']
SEEDS  = [0, 1, 2, 3, 4]
EXTRACT_POSE_IF_MISSING = False   # True hanya jika CARR/cache/pose belum ada (butuh gambar CARR, ±1-2 jam GPU)

## Persiapan (kode CARR + modul MAA)

In [ ]:
import os
os.environ['CARR_REPO'] = '/content/CARR'
os.environ['CARR_LOCK_WAIT'] = '30'          # jeda cek kunci, aman untuk Drive yang dibagi 2 akun
!git clone -q https://github.com/januaradip/CARR.git /content/CARR 2>/dev/null; git -C /content/CARR fetch -q
!git -C /content/CARR checkout -q 1193346
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
if not torch.cuda.is_available():
    raise SystemExit('Sesi ini TIDAK memakai GPU. Pilih Runtime -> Change runtime type -> T4 GPU (atau A100), '
                     'lalu Runtime -> Run all lagi. Kalau pilihan GPU tidak tersedia, kuota GPU akun ini sedang habis.')
print('GPU OK:', torch.cuda.get_device_name(0))

In [ ]:
%%writefile /content/maa.py
"""
MAA: Mobility-Aid-Aware skeleton augmentation for activity recognition on CARR.

Experiment S1 (this version): baselines without the proposed augmentation (aug 'A0') for four backbones
(ST-GCN, ST-GCN++, CTR-GCN, PoseC3D) on the nine CARR protocols (CS1-4, CL1-4, CG) and several seeds.

Builds on the public CARR benchmark code (github.com/januaradip/CARR):
  * frame index, protocol splits, run locks, trainer with checkpoint/resume  -> carr.common
  * 17 COCO keypoints (YOLOv8m-Pose), box-normalised sequences, ST-GCN / CTR-GCN -> carr.skeleton

Layout of the output (WORK):
  WORK/cache/frames.pkl, WORK/cache/seq_box.pkl         shared caches
  WORK/maa/runs/<exp>/<aug>/<model>/<split>/s<seed>/    checkpoints, DONE.json, video_probs.npz
  WORK/maa/results/<exp>_runs.csv, <exp>_summary.csv    collected results

Every unit (exp, aug, model, split, seed) is claimed with a lock file, so several Colab sessions (also from
different Google accounts sharing the same Drive folder) can work on the same experiment in parallel.
"""
import os, sys, glob, json, math, time, random, shutil, collections
import numpy as np

CARR_REPO = os.environ.get('CARR_REPO', '/content/CARR')
if CARR_REPO not in sys.path:
    sys.path.insert(0, CARR_REPO)
from carr import common as C                   # noqa: E402
from carr import skeleton as SK                # noqa: E402

MODELS = ['ST-GCN', 'ST-GCN++', 'CTR-GCN', 'PoseC3D']
SPLITS = ['CS1', 'CS2', 'CS3', 'CS4', 'CL1', 'CL2', 'CL3', 'CL4', 'CG']

CFG = dict(
    WIN=64, STRIDE=32,                         # windows as in the CARR benchmark
    EPOCHS=60, PATIENCE=12, WARMUP=3, WD=1e-4, WORKERS=2,
    BATCH={'ST-GCN': 64, 'ST-GCN++': 64, 'CTR-GCN': 64, 'PoseC3D': 32},
    ACCUM={'ST-GCN': 1, 'ST-GCN++': 1, 'CTR-GCN': 1, 'PoseC3D': 2},     # PoseC3D: effective batch 64
    LR={'ST-GCN': 1e-3, 'ST-GCN++': 1e-3, 'CTR-GCN': 1e-3, 'PoseC3D': 1e-3},
    HM_SIZE=56, HM_SIGMA=0.6, HM_RANGE=0.75, HM_FRAMES=32,              # PoseC3D heatmap volumes
)


# =============================================================================== data
def paths(data, work):
    return C.Paths(data, work)


def load_seq(P, F, pose_dir=None):
    """Box-normalised keypoint sequences {video: (T,17,3)}, cached once in WORK/cache/seq_box.pkl.
    pose_dir: folder with <video>.npz from the CARR benchmark (default WORK/cache/pose)."""
    cache = f'{P.cache}/seq_box.pkl'
    if os.path.exists(cache):
        obj = C.load_pickle(cache)
        if obj.get('sig') == P.signature() and len(obj['seq']) == len(F['videos']):
            return obj['seq']
    pose_dir = pose_dir or f'{P.cache}/pose'
    missing = [v for v in F['videos'] if not os.path.exists(f'{pose_dir}/{v}.npz')]
    if missing:
        raise FileNotFoundError(
            f'{len(missing)} of {len(F["videos"])} pose files missing in {pose_dir} (e.g. {missing[:3]}). '
            f'Set POSE_DIR to the pose cache of the CARR benchmark (CARR_work/cache/pose) or run the '
            f'pose-extraction cell.')
    C.log('Building box-normalised sequences (once)...')
    # carr.skeleton.load_sequences reads <cache>/pose/<video>.npz
    if os.path.basename(pose_dir.rstrip('/')) != 'pose':
        raise ValueError('POSE_DIR must be a folder named "pose" (e.g. .../CARR_work/cache/pose)')
    q = type('Q', (), {})()
    q.cache = os.path.dirname(pose_dir.rstrip('/'))
    seq = SK.load_sequences(q, F)
    C.save_pickle(dict(sig=P.signature(), seq=seq), cache)
    C.log(f'{len(seq)} sequences cached in {cache}')
    return seq


def make_windows(videos, seq):
    return SK.make_windows(videos, seq, None)


# ------------------------------------------------------------------ augmentations
def base_aug(x):
    """Augmentation of the CARR benchmark (kept for every variant): flip, rotation +-10 deg, scale 0.9-1.1."""
    if random.random() < 0.5:
        x = x[:, SK.FLIP]
        x[:, :, 0] *= -1
    a = math.radians(random.uniform(-10, 10))
    sc = random.uniform(0.9, 1.1)
    R = np.array([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]], np.float32) * sc
    x[:, :, :2] = x[:, :, :2] @ R.T
    return x


AUGS = {
    'A0': lambda x: x,          # baseline: base augmentation only
    # A1-A4 (generic) and U1-U5 (proposed) are added in the next notebook version
}


class SkelDS:
    def __init__(self, items, seq, F, train, aug='A0'):
        self.items, self.seq, self.F, self.train, self.aug = items, seq, F, train, AUGS[aug]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        import torch
        v, idx = self.items[i]
        x = self.seq[v][idx].copy()                    # (T,17,3): x, y (box-normalised), confidence
        if self.train:
            x = self.aug(base_aug(x))
        x = torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)).permute(2, 0, 1).contiguous()
        return x, self.F['videos'][v]['cls'], i


# =============================================================================== models
def build(name):
    import torch
    import torch.nn as nn
    if name in ('ST-GCN', 'CTR-GCN'):
        return SK.build_model(name)

    A0 = torch.tensor(SK.graph_A())                   # (3,17,17)

    class DataBN(nn.Module):
        def __init__(s, C=3, V=17):
            super().__init__()
            s.bn = nn.BatchNorm1d(C * V)

        def forward(s, x):
            N, Cc, T, V = x.shape
            x = x.permute(0, 3, 1, 2).reshape(N, V * Cc, T)
            return s.bn(x).reshape(N, V, Cc, T).permute(0, 2, 3, 1).contiguous()

    # ------------------------------------------------ ST-GCN++ (PYSKL: adaptive 'init' graph + multi-scale TCN)
    class AGCN(nn.Module):
        def __init__(s, cin, cout):
            super().__init__()
            s.K = A0.shape[0]
            s.A = nn.Parameter(A0.clone())             # initialised with the skeleton graph, fully learnable
            s.conv = nn.Conv2d(cin, cout * s.K, 1)
            s.bn = nn.BatchNorm2d(cout)
            s.res = nn.Identity() if cin == cout else nn.Sequential(nn.Conv2d(cin, cout, 1), nn.BatchNorm2d(cout))
            s.relu = nn.ReLU(inplace=True)

        def forward(s, x):
            N, _, T, V = x.shape
            y = s.conv(x).view(N, s.K, -1, T, V)
            y = torch.einsum('nkctv,kwv->nctw', y, s.A)
            return s.relu(s.bn(y) + s.res(x))

    class MSTCN(nn.Module):
        def __init__(s, cin, cout, stride=1):
            super().__init__()
            nb = 6
            mids = [cout // nb] * (nb - 1) + [cout - (cout // nb) * (nb - 1)]
            br = []
            for d, m in zip((1, 2, 3, 4), mids[:4]):
                br.append(nn.Sequential(nn.Conv2d(cin, m, 1), nn.BatchNorm2d(m), nn.ReLU(inplace=True),
                                        nn.Conv2d(m, m, (3, 1), (stride, 1), (d, 0), (d, 1))))
            br.append(nn.Sequential(nn.Conv2d(cin, mids[4], 1), nn.BatchNorm2d(mids[4]), nn.ReLU(inplace=True),
                                    nn.MaxPool2d((3, 1), (stride, 1), (1, 0))))
            br.append(nn.Conv2d(cin, mids[5], 1, (stride, 1)))
            s.br = nn.ModuleList(br)
            s.out = nn.Sequential(nn.BatchNorm2d(cout), nn.ReLU(inplace=True), nn.Conv2d(cout, cout, 1),
                                  nn.BatchNorm2d(cout))

        def forward(s, x):
            return s.out(torch.cat([b(x) for b in s.br], 1))

    class PPBlock(nn.Module):
        def __init__(s, cin, cout, stride=1, residual=True):
            super().__init__()
            s.gcn, s.tcn = AGCN(cin, cout), MSTCN(cout, cout, stride)
            if not residual:
                s.res = lambda x: 0
            elif cin == cout and stride == 1:
                s.res = nn.Identity()
            else:
                s.res = nn.Sequential(nn.Conv2d(cin, cout, 1, (stride, 1)), nn.BatchNorm2d(cout))
            s.relu = nn.ReLU(inplace=True)

        def forward(s, x):
            return s.relu(s.tcn(s.gcn(x)) + s.res(x))

    class STGCNpp(nn.Module):
        def __init__(s):
            super().__init__()
            s.bn = DataBN()
            cfg = [(3, 64, 1, False), (64, 64, 1, True), (64, 64, 1, True), (64, 64, 1, True),
                   (64, 128, 2, True), (128, 128, 1, True), (128, 128, 1, True),
                   (128, 256, 2, True), (256, 256, 1, True), (256, 256, 1, True)]
            s.blocks = nn.ModuleList(PPBlock(*c) for c in cfg)
            s.drop = nn.Dropout(0.1)
            s.fc = nn.Linear(256, 6)

        def forward(s, x):
            x = s.bn(x)
            for b in s.blocks:
                x = b(x)
            return s.fc(s.drop(x.mean((2, 3))))

    # ------------------------------------------------ PoseC3D (PYSKL: SlowOnly-R50 pose variant on joint heatmaps)
    class Bottleneck3d(nn.Module):
        def __init__(s, cin, planes, stride_s=1, inflate=False):
            super().__init__()
            kt = 3 if inflate else 1
            s.c1 = nn.Sequential(nn.Conv3d(cin, planes, (kt, 1, 1), padding=(kt // 2, 0, 0), bias=False),
                                 nn.BatchNorm3d(planes), nn.ReLU(inplace=True))
            s.c2 = nn.Sequential(nn.Conv3d(planes, planes, (1, 3, 3), (1, stride_s, stride_s), (0, 1, 1), bias=False),
                                 nn.BatchNorm3d(planes), nn.ReLU(inplace=True))
            s.c3 = nn.Sequential(nn.Conv3d(planes, planes * 4, 1, bias=False), nn.BatchNorm3d(planes * 4))
            s.down = None
            if stride_s != 1 or cin != planes * 4:
                s.down = nn.Sequential(nn.Conv3d(cin, planes * 4, 1, (1, stride_s, stride_s), bias=False),
                                       nn.BatchNorm3d(planes * 4))
            s.relu = nn.ReLU(inplace=True)

        def forward(s, x):
            r = x if s.down is None else s.down(x)
            return s.relu(s.c3(s.c2(s.c1(x))) + r)

    class PoseC3D(nn.Module):
        """Input (N,3,T,17) box-normalised keypoints -> joint heatmap volume (N,17,T/2,56,56) built on the GPU."""

        def __init__(s):
            super().__init__()
            S = CFG['HM_SIZE']
            g = torch.linspace(-CFG['HM_RANGE'], CFG['HM_RANGE'], S)
            s.register_buffer('gx', g.view(1, 1, 1, 1, S))
            s.register_buffer('gy', g.view(1, 1, 1, S, 1))
            s.sig = CFG['HM_SIGMA'] * (2 * CFG['HM_RANGE'] / (S - 1))     # sigma in grid cells -> coordinates
            s.stem = nn.Sequential(nn.Conv3d(17, 32, (1, 7, 7), padding=(0, 3, 3), bias=False),
                                   nn.BatchNorm3d(32), nn.ReLU(inplace=True))
            layers, cin = [], 32
            for planes, n, inflate in ((32, 4, False), (64, 6, True), (128, 3, True)):
                for i in range(n):
                    layers.append(Bottleneck3d(cin, planes, 2 if i == 0 else 1, inflate))
                    cin = planes * 4
            s.body = nn.Sequential(*layers)
            s.drop = nn.Dropout(0.5)
            s.fc = nn.Linear(cin, 6)

        def heatmaps(s, x):
            T = x.shape[2]
            step = max(1, T // CFG['HM_FRAMES'])
            x = x[:, :, ::step][:, :, :CFG['HM_FRAMES']].float()       # (N,3,T',17)
            u = x[:, 0].permute(0, 2, 1)[..., None, None]               # (N,17,T',1,1)
            v = x[:, 1].permute(0, 2, 1)[..., None, None]
            c = x[:, 2].permute(0, 2, 1)[..., None, None]
            return c * torch.exp(-((s.gx - u) ** 2 + (s.gy - v) ** 2) / (2 * s.sig ** 2))

        def forward(s, x):
            h = s.heatmaps(x)
            return s.fc(s.drop(s.body(s.stem(h)).mean((2, 3, 4))))

    return {'ST-GCN++': STGCNpp, 'PoseC3D': PoseC3D}[name]()


def n_params(m):
    return sum(p.numel() for p in m.parameters())


# =============================================================================== units
def units(exp, augs, models, splits, seeds):
    return [dict(exp=exp, aug=a, model=m, split=s, seed=int(sd))
            for a in augs for m in models for s in splits for sd in seeds]


def unit_dir(P, u):
    return f"{P.work}/maa/runs/{u['exp']}/{u['aug']}/{u['model']}/{u['split']}/s{u['seed']}"


def unit_tag(u):
    return f"{u['exp']} {u['aug']} {u['model']} {u['split']} s{u['seed']}"


def evaluate(rdir, model, test_ds, F, cfg):
    """Video-level macro-F1 (mean of window probabilities per video), overall and per group; saves the
    per-video probabilities for the paired bootstrap."""
    import torch
    from torch.utils.data import DataLoader
    model.load_state_dict(torch.load(f'{rdir}/best.pt', map_location='cpu', weights_only=False)['model'])
    model.to('cuda')
    dl = DataLoader(test_ds, batch_size=cfg['batch'], shuffle=False, num_workers=cfg['workers'], pin_memory=True)
    yt, yp, probs, idx = C.predict(model, dl, 'cuda', True)
    wv = np.array([test_ds.items[i][0] for i in idx])
    vids = sorted(set(wv))
    vp = np.stack([probs[wv == v].mean(0) for v in vids])
    vy = np.array([F['videos'][v]['cls'] for v in vids])
    grp = np.array([F['videos'][v]['group'] for v in vids])
    own = np.array([F['videos'][v]['owner'] for v in vids])
    np.savez_compressed(f'{rdir}/video_probs.npz', videos=np.array(vids), y=vy, probs=vp, group=grp, owner=own)
    res = {}
    for tag, m in (('all', np.ones(len(vids), bool)), ('dis', grp == 'dis'), ('nondis', grp == 'nondis')):
        if m.sum():
            r = C.classification_metrics(vy[m], vp[m].argmax(1))
            res[f'vid_f1_{tag}'], res[f'vid_acc_{tag}'], res[f'n_{tag}'] = r['macro_f1'], r['acc'], int(m.sum())
            if tag == 'all':
                C.save_json(r['cm'], f'{rdir}/confusion_video.json')
    r = C.classification_metrics(yt, yp)
    res['win_f1_all'], res['win_acc_all'] = r['macro_f1'], r['acc']
    return res


def run_unit(P, F, seq, u):
    import torch
    rdir = unit_dir(P, u)
    C.seed_everything(u['seed'])
    torch.backends.cudnn.benchmark = True
    parts = C.load_split(P, u['split'])
    tr, va, te = (make_windows(parts[k], seq) for k in ('train', 'val', 'test'))
    m = u['model']
    cfg = dict(epochs=CFG['EPOCHS'], lr=CFG['LR'][m], wd=CFG['WD'], batch=CFG['BATCH'][m], accum=CFG['ACCUM'][m],
               patience=CFG['PATIENCE'], workers=CFG['WORKERS'], warmup=CFG['WARMUP'], amp=True)
    C.log(f'windows train {len(tr)} / val {len(va)} / test {len(te)}')
    t0 = time.time()
    C.train_classifier(rdir, build(m), SkelDS(tr, seq, F, True, u['aug']), SkelDS(va, seq, F, False), cfg)
    res = evaluate(rdir, build(m), SkelDS(te, seq, F, False), F, cfg)
    hist = json.load(open(f'{rdir}/history.json')) if os.path.exists(f'{rdir}/history.json') else []
    res.update(u, params=n_params(build(m)), epochs=len(hist),
               best_val_f1=max([h['val_f1'] for h in hist], default=None),
               gpu=torch.cuda.get_device_name(0), minutes=round((time.time() - t0) / 60, 1))
    C.save_json(res, f'{rdir}/DONE.json')
    C.log(f"{unit_tag(u)} finished: video macro-F1 {res['vid_f1_all']:.4f}"
          + (f" (dis {res['vid_f1_dis']:.4f})" if 'vid_f1_dis' in res else ''))


def execute(P, F, seq, todo, order='forward', session_seed=None):
    todo = list(todo)
    if order == 'reverse':
        todo = todo[::-1]
    elif order == 'shuffle':
        random.Random(session_seed if session_seed is not None else C.SESSION).shuffle(todo)
    busy = []
    for u in todo:
        rdir = unit_dir(P, u)
        if os.path.exists(f'{rdir}/DONE.json'):
            continue
        if not C.claim_run(rdir):
            busy.append(unit_tag(u))
            continue
        C.log(f'===== {unit_tag(u)} =====')
        try:
            run_unit(P, F, seq, u)
        finally:
            C.release_run(rdir)
    left = [u for u in todo if not os.path.exists(f'{unit_dir(P, u)}/DONE.json')]
    C.log(f'Session finished. Open units: {len(left)} (in progress elsewhere: {len(busy)}).')
    return left


def status(P, todo):
    import pandas as pd
    rows = []
    for u in todo:
        d = unit_dir(P, u)
        st = 'done' if os.path.exists(f'{d}/DONE.json') else ('running' if os.path.exists(f'{d}/LOCK') else
                                                              ('started' if os.path.exists(f'{d}/history.json') else 'open'))
        rows.append(dict(u, status=st))
    df = pd.DataFrame(rows)
    return df.pivot_table(index=['aug', 'model'], columns='status', values='seed', aggfunc='count', fill_value=0)


# =============================================================================== results
def collect(P, exp):
    import pandas as pd
    rows = []
    for f in glob.glob(f'{P.work}/maa/runs/{exp}/*/*/*/*/DONE.json'):
        with open(f) as fh:
            rows.append(json.load(fh))
    if not rows:
        C.log('No finished units yet.')
        return None
    df = pd.DataFrame(rows)
    os.makedirs(f'{P.work}/maa/results', exist_ok=True)
    df.to_csv(f'{P.work}/maa/results/{exp}_runs.csv', index=False)
    return df


def summary(P, exp):
    """Video macro-F1 (%) per model and protocol: CS (mean of 4 folds), CS-dis, CL, CG; mean +- sd over seeds."""
    import pandas as pd
    df = collect(P, exp)
    if df is None:
        return None
    df['protocol'] = df['split'].str.replace(r'\d+$', '', regex=True)
    out = []
    for (aug, model), g in df.groupby(['aug', 'model']):
        row = dict(aug=aug, model=model)
        for prot, col, name in (('CS', 'vid_f1_all', 'CS'), ('CS', 'vid_f1_dis', 'CS-dis'),
                                ('CL', 'vid_f1_all', 'CL'), ('CG', 'vid_f1_all', 'CG')):
            h = g[g.protocol == prot]
            if h.empty or col not in h:
                continue
            per_seed = h.groupby('seed')[col].mean() * 100          # fold mean per seed
            nf = h.groupby('seed')['split'].nunique()
            full = nf.max() == (1 if prot == 'CG' else 4)
            row[name] = f"{per_seed.mean():.1f} ± {per_seed.std(ddof=1) if len(per_seed) > 1 else 0:.1f}" \
                        f" (n={len(per_seed)}{'' if full else ', incomplete'})"
        row['params_M'] = round(g['params'].iloc[0] / 1e6, 2)
        row['min_per_run'] = round(g['minutes'].median(), 1)
        out.append(row)
    s = pd.DataFrame(out)
    s.to_csv(f'{P.work}/maa/results/{exp}_summary.csv', index=False)
    return s


In [ ]:
import sys, importlib
sys.path.insert(0, '/content')
import maa; importlib.reload(maa)
from carr import common as C

P = maa.paths(DATA, DATA)                    # cache/ dan hasil di dalam folder CARR, sama seperti benchmark
F = C.load_frames(P)
POSE_DIR = f'{P.cache}/pose'
if EXTRACT_POSE_IF_MISSING:
    !pip install -q ultralytics
    from carr import skeleton as SK
    SK.extract_pose(P, F)
seq = maa.load_seq(P, F, POSE_DIR)
print(f"{len(F['videos'])} video, {len(seq)} sekuens keypoint")
for s in ('CS1', 'CG'):
    p = C.load_split(P, s)
    print(s, {k: len(v) for k, v in p.items()},
          '| test alat bantu:', sum(F['videos'][v]['group'] == 'dis' for v in p['test']))

## Jalankan unit

In [ ]:
TODO = maa.units(EXP, AUGS, MODELS, SPLITS, SEEDS)
print(len(TODO), 'unit dipilih')
left = maa.execute(P, F, seq, TODO, order='shuffle')      # urutan acak per sesi

## Status dan ringkasan
Bisa dijalankan kapan saja, dari sesi mana saja.

In [ ]:
ALL = maa.units(EXP, AUGS, ['ST-GCN', 'ST-GCN++', 'CTR-GCN', 'PoseC3D'], SPLITS, SEEDS)
display(maa.status(P, ALL))
s = maa.summary(P, EXP)
display(s)
print('Disimpan di', f'{WORK}/maa/results/')